# Part 6 — Anti-Hallucination Fine-tuning

**Model:** MedGemma-1.5-4B fine-tuned with **RLFR** (Reinforcement Learning from Feature Rewards).  
A frozen hallucination-detection probe served as the reward signal,  
reducing hallucination rates by ~70% on hard medical questions.

| Metric | Vanilla MedGemma | This Model |
|--------|-----------------|------------|
| MedHallu HARD halluc rate | 51.5% | **13.5%** |

This model is used by the **Doc Agent** (Part 7) to generate pharmacovigilance documents  
with reduced risk of fabricating clinical details.

**Requirements:** GPU with >= 16 GB VRAM

---

This notebook is part of the **MedGemma Clinical Trial Engine** pipeline:

```
Part 1  Visual AE Detection ─── MedGemma 1.5 + MedSigLIP (image → AE classification)
Part 2  Cough Detection ──────── HeAR + 2-Stage Classifier (audio → cough type)
Part 3  Care AI Conversation ── MedGemma-4B as virtual nurse (multi-turn dialogue → AE detection)
    ↓
Part 4  Rule Set Generation ─── 10 biomedical DBs → LLM synthesis → simulation parameters
Part 5  Simulation Pipeline ─── Hazard functions + LLM enrichment → synthetic clinical trial data
    ↓
Part 6  Anti-Hallucination ──── RLFR fine-tuning to reduce fabrication in medical text
Part 7  Doc Agent ──────────────  CRF data → MedWatch 3500A pharmacovigilance reports
```


## 1. Install Dependencies

In [ ]:
# Run once to install dependencies
!pip install -q torch transformers accelerate

In [1]:
# Dependencies: torch, transformers, accelerate (assumed pre-installed in .venv)

## 2. Load Model

> **Tokenizer EOS fix (important):** MedGemma's tokenizer reports `eos_token_id=1` (`<eos>`), but the model actually generates `<end_of_turn>` (token id 106) to signal completion. Without patching this, `model.generate()` never sees the real stop token and runs until `max_new_tokens` every time — producing garbled, repetitive output. The fix below overrides `eos_token` to `<end_of_turn>` so generation stops correctly.

In [2]:
import os
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

# Local model path (use HuggingFace ID when published)
MODEL_ID = os.environ.get("ANTIHALLU_MODEL_ID", "/data2/workspace/vital/models/medgemma-4b-ft-antihallu")

GPU_ID = int(os.environ.get("GPU_ID", "7"))
device = f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(GPU_ID)}")

In [3]:
print(f"Downloading & loading {MODEL_ID} ...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# MedGemma terminates with <end_of_turn> (id=106), not <eos> (id=1)
eot_id = tokenizer.convert_tokens_to_ids('<end_of_turn>')
if eot_id is not None and eot_id != tokenizer.unk_token_id:
    tokenizer.eos_token = '<end_of_turn>'
    tokenizer.eos_token_id = eot_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": GPU_ID},
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.eval()

print(f"Loaded in {time.time()-t0:.1f}s ({sum(p.numel() for p in model.parameters())/1e9:.1f}B params)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loaded in 7.1s (4.3B params)


## 3. Generate Medical Answers

In [4]:
def ask(question, temperature=0.0, max_new_tokens=256):
    """Ask a medical question and get an answer."""
    prompt = (
        f"<start_of_turn>user\n"
        f"Answer the following medical question concisely and accurately:\n\n{question}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

In [5]:
questions = [
    "What is metformin used for?",
    "What are the common side effects of aspirin?",
    "What is the Glasgow Coma Scale?",
    "What causes Addison's disease?",
    "What is the difference between Crohn's disease and ulcerative colitis?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

Q: What is metformin used for?


A: Metformin is a medication primarily used to treat type 2 diabetes. It works by:

* **Reducing glucose production** by the liver.
* **Increasing insulin sensitivity** in the body's cells, allowing them to take up glucose more effectively.

It is often the first-line medication prescribed for type 2 diabetes.

Q: What are the common side effects of aspirin?


A: Aspirin can cause a variety of side effects, some common and others less frequent. Common side effects include:

*   **Gastrointestinal issues:** Stomach pain, heartburn, nausea, indigestion, and ulcers.
*   **Bleeding:** Increased risk of bleeding, including nosebleeds, easy bruising, and prolonged bleeding from cuts.
*   **Reye's syndrome:** A rare but serious condition in children and teenagers recovering from viral infections.
*   **Allergic reactions:** Rash, itching, hives, and difficulty breathing.
*   **Tinnitus:** Ringing in the ears.
*   **Headache.**
*   **Dizziness.**

It's important to consult a healthcare professional for any concerns about aspirin side effects.

Q: What is the Glasgow Coma Scale?


A: The Glasgow Coma Scale (GCS) is a neurological scale used to assess a person's level of consciousness after a head injury. It measures three components:

1.  **Eye Opening:** How the person opens their eyes (spontaneously, to speech, to pain, or not).
2.  **Verbal Response:** The type of words the person uses (oriented, confused, incomprehensible, or not verbal).
3.  **Motor Response:** The person's movements (no response, abnormal flexion, abnormal extension, or withdrawal).

Each component is scored from 1 to 6, and the total score ranges from 3 to 15. A lower score indicates a more severe level of unconsciousness.

The GCS is a quick and standardized way for healthcare professionals to objectively assess a patient's neurological status and monitor their condition over time.

Q: What causes Addison's disease?


A: Addison's disease is caused by the **destruction of the adrenal glands**, which are small glands located on top of the kidneys. This destruction leads to a deficiency of hormones produced by the adrenal glands, primarily **cortisol** and **aldosterone**.

The exact cause of this destruction is often unknown, but it can be due to:

* **Autoimmune disorders:** The body's immune system mistakenly attacks the adrenal glands. This is the most common cause in developed countries.
* **Infections:** Viral infections (like tuberculosis, fungal infections, or HIV) can damage the adrenal glands.
* **Adrenal hemorrhage:** Bleeding into the adrenal glands.
* **Adrenal tumors:** Cancerous or non-cancerous tumors can destroy the adrenal glands.
* **Genetic disorders:** Rare inherited conditions affecting the adrenal glands.
* **Adrenal destruction:** Certain medications or medical treatments can damage the adrenal glands.

It's important to consult a healthcare professional for accurate diagnosis 

A: Crohn's disease and ulcerative colitis are both inflammatory bowel diseases (IBD) that can cause inflammation and ulcers in the digestive tract. However, they differ in several key ways:

* **Location of Inflammation:** Crohn's disease can affect any part of the digestive tract from the mouth to the anus, often in a patchy pattern. Ulcerative colitis primarily affects the colon (large intestine) and rectum, and inflammation is typically continuous.
* **Depth of Inflammation:** Crohn's disease often involves deeper layers of the bowel wall (transmural inflammation). Ulcerative colitis usually involves only the innermost lining (mucosa).
* **Symptoms:** While both can cause diarrhea, abdominal pain, and weight loss, Crohn's disease may also cause fistulas (abnormal tunnels) and strictures (narrowing of the bowel). Ulcerative colitis is more commonly associated with bloody diarrhea and rectal bleeding.
* **Complications:** Both can lead to complications like strictures, fistulas, and i

## 4. Try Your Own

In [6]:
print(ask("What are the contraindications for thrombolytic therapy in acute stroke?"))

Contraindications for thrombolytic therapy (like tPA) in acute ischemic stroke include:

*   **Active internal bleeding:** This is the most critical contraindication.
*   **Recent major surgery or serious trauma:** Within the last 3-4 weeks.
*   **History of intracranial hemorrhage (ICH):** Within the last 3 months.
*   **Known intracranial neoplasm (tumor):** Within the last 2 years.
*   **Known structural cerebral vascular lesion:** Such as AVM or aneurysm.
*   **Known papilledema:** Indicating increased intracranial pressure.
*   **Recent gastrointestinal (GI) or urinary tract hemorrhage:** Within the last 2 months.
*   **Known bleeding diathesis:** Such as platelet count < 100,000/mm³, INR > 1.7, or aPTT > 40 seconds.
*   **Use of anticoagulants:** Such as warfarin (INR > 1.7) or direct thrombin inhibitors/factor Xa inhibitors (depending on specific drug and timing).
*   **Severe uncontrolled hypertension:** Systolic BP > 185 mmHg or diastolic BP > 110 mmHg at the time of
